# Innovator Patent Domain Adapter — Kaggle free-tier run

**Purpose:** a bounded patent-language LoRA adapter using the openly licensed AllenAI / PatentsView allenai/us-patents corpus. This is a U.S. patent corpus, not a WIPO PATENTSCOPE mirror.

**WIPO data boundary:** WIPO's PCT-data products are license-governed. Its non-derivative license permits acquiring, downloading and storing the complete dataset in a searchable database; bulk redistribution requires a derivative license with added value. The PATENTSCOPE web service limits retrieval to 10 actions per minute per subscriber/IP and prohibits parallel IP use to increase throughput. This notebook does not scrape PATENTSCOPE; use an appropriately licensed WIPO data product for a full PCT corpus. Terms: https://www.wipo.int/en/web/patentscope/data/terms .

**Dataset:** approximately 7.8M train + 79,836 validation U.S. patent documents (1976–2025), 124 GB total, ODC-BY, attribution to PatentsView. This notebook streams 12,000 train + 600 valid examples; it does not download the full corpus. Dataset card: https://huggingface.co/datasets/allenai/us-patents .

**Compute:** Qwen2.5-0.5B-Instruct + LoRA, 1,500 optimizer steps, Kaggle GPU required. No paid APIs and no automatic Hub upload.

**Output:** LoRA adapter, tokenizer, README, provenance manifest and SHA-256 hashes. Research candidate only—not a patentability, novelty, legal, or invention-quality system.


In [ ]:
!pip -q install transformers==4.48.3 peft==0.14.0 accelerate==1.2.1 datasets==3.2.0

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
import re,io,json,time,hashlib,zipfile,tarfile,pathlib,random,requests,xml.etree.ElementTree as ET
from datetime import datetime,timezone
from urllib.parse import quote
import torch
SEED=42; random.seed(SEED); torch.manual_seed(SEED)
os.environ['TOKENIZERS_PARALLELISM']='false'
if not torch.cuda.is_available(): raise RuntimeError("Select a Kaggle GPU accelerator first.")
print(torch.cuda.get_device_name(0))
WORK=pathlib.Path("/kaggle/working/patent_adapter"); WORK.mkdir(parents=True,exist_ok=True)
MAX_PATENTS=12000; MAX_STEPS=1500
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
BASE="https://bulkdata.uspto.gov/v1/BDSS-API"
HEADERS={"User-Agent":"Innovator-Kaggle-Patent-Research/1.0","Accept":"application/json"}

## 1. Stream an open, attributable USPTO patent corpus

WIPO PATENTSCOPE forbids automated queries, scraping, bulk acquisition and bulk downloads. This notebook instead uses the AllenAI allenai/us-patents dataset, which contains about 7.8M U.S. patent grants/applications (1976–2025), is licensed ODC-BY, and requires PatentsView attribution. It streams a bounded sample rather than downloading the 124 GB corpus. Dataset card: https://huggingface.co/datasets/allenai/us-patents


In [ ]:
from datasets import load_dataset

DATASET_ID='allenai/us-patents'
TRAIN_LIMIT=12000; VALID_LIMIT=600

def collect(split,limit):
    stream=load_dataset(DATASET_ID,split=split,streaming=True)
    rows=[]
    for row in stream:
        text=row.get('text')
        if not isinstance(text,str) or len(text)<220: continue
        rows.append({'text':text[:12000],'corpus_id':row.get('corpus_id'),'filing_date':row.get('filing_date'),'patent_type':row.get('patent_type'),'source':DATASET_ID})
        if len(rows)>=limit: break
    return rows

patents=collect('train',TRAIN_LIMIT)
validation_patents=collect('validation',VALID_LIMIT)
if len(patents)<1000 or len(validation_patents)<100: raise RuntimeError('Insufficient streamed patent examples; verify dataset access and splits.')
source={'provider':'AllenAI Common Pile / PatentsView','dataset_id':DATASET_ID,'dataset_card':'https://huggingface.co/datasets/allenai/us-patents','license':'ODC-BY','attribution':'PatentsView','train_split':'train','validation_split':'validation','train_rows':len(patents),'validation_rows':len(validation_patents),'retrieved_utc':datetime.now(timezone.utc).isoformat(),'wipo_patentscope_scraping':False}
def digest_text(rows): return hashlib.sha256(''.join(r['text'] for r in rows).encode()).hexdigest()
source['sample_train_text_sha256']=digest_text(patents); source['sample_validation_text_sha256']=digest_text(validation_patents)
(WORK/'source_manifest.json').write_text(json.dumps(source,indent=2))
(WORK/'patents.jsonl').write_text(''.join(json.dumps(x)+'\n' for x in patents))
print(source)


## 2. Stream bounded train/validation samples and preserve provenance

The dataset provides a time-separated validation split (2025-01-14 to 2025-09-19). The notebook streams a bounded sample and does not download the full 124 GB corpus.


In [ ]:
train_rows=patents
eval_rows=validation_patents
print('Train rows:',len(train_rows),'validation rows:',len(eval_rows))
print('Sample record:',{k:(v[:400] if isinstance(v,str) else v) for k,v in train_rows[0].items()})


## 3. Train a Qwen 0.5B LoRA adapter

This is a bounded continued-domain-adaptation experiment, not foundation-model pretraining. The time-separated validation split is kept separate from training.


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,DataCollatorForLanguageModeling,Trainer,TrainingArguments
from peft import LoraConfig,get_peft_model,TaskType
train_rows=patents; eval_rows=validation_patents
tok=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
def tokenize(batch): return tok(batch["text"],truncation=True,max_length=512)
raw_train=Dataset.from_list(train_rows); train_ds=raw_train.map(tokenize,batched=True,remove_columns=raw_train.column_names)
raw_eval=Dataset.from_list(eval_rows); eval_ds=raw_eval.map(tokenize,batched=True,remove_columns=raw_eval.column_names)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16,device_map={"":0})
model.config.use_cache=False
cfg=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,bias="none",task_type=TaskType.CAUSAL_LM,target_modules=["q_proj","k_proj","v_proj","o_proj"])
model=get_peft_model(model,cfg); model.print_trainable_parameters()
args=TrainingArguments(output_dir=str(WORK/"trainer_runs"),num_train_epochs=1,max_steps=MAX_STEPS,per_device_train_batch_size=2,per_device_eval_batch_size=2,gradient_accumulation_steps=4,learning_rate=2e-4,fp16=True,logging_steps=50,save_strategy="steps",save_steps=1500,eval_strategy="steps",eval_steps=1500,save_total_limit=2,gradient_checkpointing=False,report_to="none",dataloader_num_workers=0,optim="adamw_torch")
trainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=eval_ds,data_collator=DataCollatorForLanguageModeling(tok,mlm=False),processing_class=tok)
train_result=trainer.train(); eval_result=trainer.evaluate()
ADAPTER=WORK/"patent_lora_adapter"; trainer.model.save_pretrained(ADAPTER); tok.save_pretrained(ADAPTER)
print("Train:",train_result.metrics); print("Eval:",eval_result)

## 4. Generate a sample and save a reproducibility manifest

The held-out split is a random 5% of one bounded source archive. Loss does not establish novelty detection, citation quality, legal accuracy, or patentability. Keep the adapter as an unpromoted research candidate until a separate evaluation is designed.

In [ ]:
prompt='Title: A low-energy water purification device\nAbstract:'
inputs=tok(prompt,return_tensors='pt').to(model.device)
with torch.no_grad(): output=model.generate(**inputs,max_new_tokens=100,do_sample=False,pad_token_id=tok.eos_token_id)
print(tok.decode(output[0],skip_special_tokens=True))
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
hashes={p.name:sha(p) for p in ADAPTER.iterdir() if p.is_file()}
manifest={'base_model':MODEL_ID,'task':'patent-domain LoRA adaptation','source':source,'patent_records':len(patents),'train_records':len(train_rows),'eval_records':len(eval_rows),'max_steps':MAX_STEPS,'seed':SEED,'train_metrics':train_result.metrics,'eval_metrics':eval_result,'adapter_sha256':hashes,'status':'research_candidate_not_promoted','limitations':['U.S. patent corpus, not WIPO PATENTSCOPE','Bounded streaming sample, not all global patents','No novelty/legal/patentability benchmark']}
(WORK/'training_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
print('Saved adapter:',ADAPTER,'manifest:',WORK/'training_manifest.json')


In [ ]:
import zipfile
README = '''# Innovator Patent Domain Adapter (research candidate)

- Base model: Qwen/Qwen2.5-0.5B-Instruct
- Method: LoRA continued domain adaptation
- Corpus: allenai/us-patents (PatentsView), ODC-BY; see training_manifest.json
- Scope: bounded streamed subset, not the full 124 GB corpus
- WIPO PCT-data products are license-governed; full dataset acquisition/storage requires a non-derivative license and bulk redistribution requires a derivative license. PATENTSCOPE service retrieval is rate-limited to 10 actions per minute per subscriber/IP; this notebook does not scrape it.
- Status: research candidate, not promoted

This adapter is not a novelty, patentability, legal, or scientific-validity system. Validate retrieval, citations, claims, and expert review before use.
'''
(WORK/'README.md').write_text(README)
BUNDLE=WORK/'innovator_patent_adapter_artifacts.zip'
with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as z:
    for path in ADAPTER.rglob('*'):
        if path.is_file(): z.write(path,arcname=path.relative_to(WORK))
    for path in [WORK/'README.md',WORK/'source_manifest.json',WORK/'training_manifest.json']:
        z.write(path,arcname=path.relative_to(WORK))
print('Artifact bundle:',BUNDLE,'bytes:',BUNDLE.stat().st_size)
